# Multimodal Agent - Test Notebook

This notebook tests the Strands Agent with AgentCore Memory **before deployment**.

It validates that:
1. The agent can process **text** messages
2. The agent can process **images** and store the understanding as text in memory
3. The agent can process **documents** (PDF) and store summaries in memory
4. The agent can handle **audio transcripts**
5. The agent can analyze **videos** using TwelveLabs Pegasus via Amazon Bedrock
6. Memory persists across turns - the agent can answer questions about previously shared multimedia

> **Prerequisites**: Install dependencies with `uv pip install strands-agents strands-agents-tools boto3 twelvelabs`

In [ ]:
%pip install strands-agents strands-agents-tools boto3 --quiet

## 1. Setup - Create the Agent (without AgentCore Memory)

For local testing, we create the agent **without** AgentCore Memory.
The memory integration will be tested separately when deployed to AgentCore Runtime.

In [ ]:
import os
import json
import base64
from pathlib import Path

import boto3
from strands import Agent, tool
from strands.models import BedrockModel


# --- Video Analysis Tool (TwelveLabs Pegasus via Bedrock) ---
@tool
def video_analysis(
    s3_uri: str,
    prompt: str = "Describe this video in detail including visual content, actions, text on screen, and any spoken words.",
    temperature: float = 0.2,
) -> dict:
    """Analyze a video using TwelveLabs Pegasus model via Amazon Bedrock.

    Args:
        s3_uri: S3 URI of the video (e.g., s3://bucket/key.mp4).
        prompt: Question or instruction about the video content.
        temperature: Model temperature for response generation.

    Returns:
        Dict with video analysis results.
    """
    aws_region = os.environ.get("AWS_REGION", "us-east-1")
    bedrock = boto3.client("bedrock-runtime", region_name=aws_region)
    sts = boto3.client("sts", region_name=aws_region)
    account_id = sts.get_caller_identity()["Account"]

    body = {
        "inputPrompt": prompt,
        "mediaSource": {"s3Location": {"uri": s3_uri, "bucketOwner": account_id}},
        "temperature": temperature,
    }

    response = bedrock.invoke_model(
        modelId="twelvelabs.pegasus-1-2-v1:0",
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )

    response_body = json.loads(response["body"].read())
    return {
        "status": "success",
        "content": [{"json": {"s3_uri": s3_uri, "analysis": response_body.get("message", "")}}],
    }


# --- System Prompt ---
SYSTEM_PROMPT = """You are a helpful assistant on WhatsApp. You can process text, images, documents, audio, and videos.

Always respond in the same language the user writes to you.

When you receive multimedia, describe or summarize the content in detail so it is preserved in your memory for future questions.

For videos, use the video_analysis tool with the provided S3 URI.

Keep responses under 4000 characters and use WhatsApp-friendly formatting.
"""

model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT, tools=[video_analysis])
print("Agent created with video_analysis tool")

## 2. Test Text Messages

Basic text conversation to verify the agent responds correctly.

In [ ]:
result = agent("Hello! What can you help me with?")
print(str(result))

## 3. Test Image Processing

Simulates sending an image to the agent. The agent should describe it in detail
so the understanding can be stored in memory (text-only).

> Replace `IMAGE_PATH` with a real image path on your system to test.

In [ ]:
def build_image_message(image_path: str, prompt: str = "Analyze this image in detail."):
    """Build a multimodal message with an image for the agent."""
    image_bytes = Path(image_path).read_bytes()

    # Detect format
    if image_bytes[:3] == b"\xff\xd8\xff":
        fmt = "jpeg"
    elif image_bytes[:8] == b"\x89PNG\r\n\x1a\n":
        fmt = "png"
    elif image_bytes[:4] == b"GIF8":
        fmt = "gif"
    elif image_bytes[:4] == b"RIFF" and image_bytes[8:12] == b"WEBP":
        fmt = "webp"
    else:
        fmt = "jpeg"

    return [
        {
            "image": {
                "format": fmt,
                "source": {"bytes": image_bytes},
            }
        },
        {
            "text": (
                f"{prompt}\n\n"
                "IMPORTANT: Provide a detailed description of what you see in this image. "
                "This description will be stored in memory so you can answer future questions "
                "about this image even when you no longer have access to it."
            ),
        },
    ]

# --- Test with a sample image ---
# Replace with an actual image path to test
IMAGE_PATH = "sample_image.jpg"  # Change this

if Path(IMAGE_PATH).exists():
    content = build_image_message(IMAGE_PATH, "What do you see in this image?")
    result = agent(content)
    print(str(result))
else:
    print(f"Image not found: {IMAGE_PATH}")
    print("Skipping image test. Place a test image and update IMAGE_PATH.")

## 4. Test Memory Recall (Follow-up Question)

After processing an image, the agent should be able to answer follow-up questions
about it using conversational context (and in production, AgentCore Memory).

In [ ]:
# Follow-up question about the previously shared image
result = agent("What were the main things you saw in the image I just sent?")
print(str(result))

## 5. Test Audio Transcript Processing

Simulates receiving a transcription from an audio message.
The agent processes the text and stores the understanding in memory.

In [ ]:
# Simulate an audio transcription
audio_transcript = "Hey, I need to schedule a meeting for next Tuesday at 3 PM with the marketing team to discuss the Q4 campaign strategy."

audio_message = [
    {
        "text": (
            f"The user sent an audio message. Here is the transcription:\n\n"
            f'"{audio_transcript}"\n\n'
            "Respond to the transcribed audio message. The transcription content "
            "will be stored in memory for future reference."
        ),
    }
]

result = agent(audio_message)
print(str(result))

## 6. Test Document Processing

Simulates sending a PDF document. The agent extracts key content and creates
a text summary for memory storage.

> Replace `DOCUMENT_PATH` with a real PDF path to test.

In [ ]:
def build_document_message(doc_path: str, prompt: str = "Analyze this document."):
    """Build a multimodal message with a document for the agent."""
    doc_bytes = Path(doc_path).read_bytes()
    ext = Path(doc_path).suffix.lstrip(".")
    name = Path(doc_path).stem

    return [
        {
            "document": {
                "format": ext,
                "name": name,
                "source": {"bytes": doc_bytes},
            }
        },
        {
            "text": (
                f"{prompt}\n\n"
                "IMPORTANT: Extract and summarize the key content from this document. "
                "This summary will be stored in memory so you can answer future questions "
                "about this document even when you no longer have access to it."
            ),
        },
    ]

# --- Test with a sample document ---
DOCUMENT_PATH = "sample_document.pdf"  # Change this

if Path(DOCUMENT_PATH).exists():
    content = build_document_message(DOCUMENT_PATH, "Summarize this document for me.")
    result = agent(content)
    print(str(result))
else:
    print(f"Document not found: {DOCUMENT_PATH}")
    print("Skipping document test. Place a test PDF and update DOCUMENT_PATH.")

## 7. Test Video Analysis (TwelveLabs Pegasus via Bedrock)

Analyzes a video using the `video_analysis` tool which calls TwelveLabs Pegasus
model via Amazon Bedrock. This provides rich visual + audio understanding,
far superior to audio-only transcription.

> **Prerequisites**: Upload a test video to S3 and update `VIDEO_S3_URI` below.
> Ensure the TwelveLabs Pegasus model (`twelvelabs.pegasus-1-2-v1:0`) is enabled in your Bedrock console.

In [ ]:
# --- Test video analysis with TwelveLabs Pegasus ---
# Replace with an actual S3 URI of a video file
VIDEO_S3_URI = "s3://your-bucket/videos/sample-video.mp4"  # Change this

# To upload a local video to S3 for testing:
# s3 = boto3.client("s3")
# s3.upload_file("local_video.mp4", "your-bucket", "videos/sample-video.mp4")

# Simulate the prompt the agent would receive when a video is sent via WhatsApp
video_prompt = (
    f"The user sent a video stored at: {VIDEO_S3_URI}\n\n"
    "User's message: What is this video about?\n\n"
    "Use the video_analysis tool with the S3 URI above to analyze "
    "the video's visual content and audio. Provide a comprehensive "
    "description that includes visual elements, actions, spoken words, "
    "and any on-screen text."
)

# The agent will automatically call the video_analysis tool
result = agent(video_prompt)
print(str(result))

## 8. Test Conversation Memory

Ask the agent about things from earlier in the conversation to verify it retains context.

In [ ]:
# Ask about the audio message from earlier
result = agent("What meeting did I ask about in my audio message?")
print(str(result))

In [ ]:
# Ask about the video from earlier
result = agent("What was the revenue increase mentioned in the video?")
print(str(result))

## 9. Simulate Full Payload (as AgentCore would receive)

This simulates the exact payload format that the Lambda handler sends to AgentCore Runtime.

In [ ]:
# Simulate text-only payload
text_payload = {
    "prompt": "What is the capital of France?"
}

# Simulate image payload (would include base64 data in production)
image_payload = {
    "prompt": "What do you see?",
    "media": {
        "type": "image",
        "format": "jpeg",
        "data": "<base64_encoded_image_data>",
    }
}

# Simulate audio transcript payload
audio_payload = {
    "prompt": "Process this audio",
    "media": {
        "type": "audio_transcript",
        "data": "The user said: please remind me to buy groceries tomorrow.",
    }
}

# Simulate video payload (TwelveLabs Pegasus via Bedrock)
video_payload = {
    "prompt": "Analyze this video in detail.",
    "media": {
        "type": "video",
        "s3_uri": "s3://bucket/videos/sample.mp4",
    }
}

# Simulate document payload
document_payload = {
    "prompt": "Summarize this document",
    "media": {
        "type": "document",
        "format": "pdf",
        "data": "<base64_encoded_pdf_data>",
        "name": "quarterly_report",
    }
}

print("Payload formats validated:")
for name, payload in [
    ("text", text_payload),
    ("image", image_payload),
    ("audio_transcript", audio_payload),
    ("video", video_payload),
    ("document", document_payload),
]:
    has_media = "media" in payload
    media_type = payload.get("media", {}).get("type", "none")
    print(f"  {name}: prompt='{payload['prompt'][:40]}...' media={has_media} type={media_type}")